# Baselines

### Setup and Imports


In [3]:
import sys
import os
import pandas as pd
import numpy as np
import json
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

# Add paths for imports
sys.path.append('../../stage1_analysis/source_finding')
sys.path.append('../../stage1_analysis/mapping_generation')

from rag_source_finder import RAGSourceFinder
from easy_llm_importer import LLMClient

# Data path
DATA_PATH = "../../data/SCAR_cleaned_manually.csv"


### Baseline 1: Pure Embedding Retrieval (Top 3)


In [4]:
# Initialize RAG source finder
finder = RAGSourceFinder(embedding_mode="name_properties")
finder.load_corpus_from_csv(DATA_PATH)
finder.embed_corpus()


Loaded 333 unique sources from corpus (mode: name_properties)
Generating embeddings for corpus...
Generated 333 embeddings


In [5]:
# Run Baseline 1: Pure embedding retrieval with top 3
baseline1_results = finder.evaluate_on_dataset(DATA_PATH, top_k=3)
baseline1_results['baseline'] = 'embedding_top3'
baseline1_results.to_csv('baseline1_embedding_top3.csv', index=False)
print(f"Baseline 1 saved: {len(baseline1_results)} rows")
baseline1_results.head()


Evaluating on 400 examples...
Processed 10/400 examples
Processed 20/400 examples
Processed 30/400 examples
Processed 40/400 examples
Processed 50/400 examples
Processed 60/400 examples
Processed 70/400 examples
Processed 80/400 examples
Processed 90/400 examples
Processed 100/400 examples
Processed 110/400 examples
Processed 120/400 examples
Processed 130/400 examples
Processed 140/400 examples
Processed 150/400 examples
Processed 160/400 examples
Processed 170/400 examples
Processed 180/400 examples
Processed 190/400 examples
Processed 200/400 examples
Processed 210/400 examples
Processed 220/400 examples
Processed 230/400 examples
Processed 240/400 examples
Processed 250/400 examples
Processed 260/400 examples
Processed 270/400 examples
Processed 280/400 examples
Processed 290/400 examples
Processed 300/400 examples
Processed 310/400 examples
Processed 320/400 examples
Processed 330/400 examples
Processed 340/400 examples
Processed 350/400 examples
Processed 360/400 examples
Process

,id,target,gold_source,all_golden_sources,predicted_rank_1,gold_rank,num_golden_found,found_golden_sources,golden_ranks,top_k_sources,top_k_scores,embedding_mode,query_embedding_text,target_background,target_properties,baseline
0,1,biological clock,clock,[clock],clock,1,1,[clock],[1],"[clock, Biological Evolution, Time Machine]","[0.48747001166636494, 0.4727238510932971, 0.43...",name_properties,"biological clock. changes, state, adjust",The biological clock is a fundamental aspect o...,"changes, state, adjust",embedding_top3
1,2,Biosphere,Library,[Library],Ecosystem,-1,0,[],[],"[Ecosystem, ecosystem, Biological Evolution]","[0.7417536552432331, 0.600835231456412, 0.4637...",name_properties,"Biosphere. biology, biodiversity, ecosystem",The biosphere refers to the sum of all ecosyst...,"biology, biodiversity, ecosystem",embedding_top3
2,3,Respiratory system,engine,"[engine, Air handling system, air circulation ...",respiration,-1,0,[],[],"[respiration, Human Body, the skeletal system]","[0.5764429137186268, 0.4766108890476132, 0.433...",name_properties,"Respiratory system. oxygen, the lungs, breathi...",The respiratory system is a critical biologica...,"oxygen, the lungs, breathing muscles",embedding_top3
3,4,Spread of Pathogens,Spread of Fire,[Spread of Fire],Crowd,3,1,[Spread of Fire],[3],"[Crowd, Disease, Spread of Fire]","[0.5004412128308713, 0.47933010292152944, 0.45...",name_properties,"Spread of Pathogens. pathogen, crowd, Preventi...","The spread of pathogens is a serious concern, ...","pathogen, crowd, Prevention and control measures",embedding_top3
4,5,Gene editing,kirigami,[kirigami],bacterial mutation,-1,0,[],[],"[bacterial mutation, Evolution, edit]","[0.37034962784168296, 0.3350655099294726, 0.32...",name_properties,"Gene editing. Gene, CRISPR-Cas9 Technology, ed...",Gene editing is a revolutionary technique in m...,"Gene, CRISPR-Cas9 Technology, edited gene",embedding_top3


## Baseline 2: Embedding Retrieval + LLM Selection (Top 10 → 3)


In [ ]:
# Initialize LLM client for Llama 405B
llm_client = LLMClient()

import re
import ast

# Build source properties lookup from SCAR data
df_scar = pd.read_csv(DATA_PATH)
source_properties_lookup = {}
for _, row in df_scar.iterrows():
    source_name = row['system_b']
    if source_name not in source_properties_lookup:
        source_properties_lookup[source_name] = set()
    # Extract source properties from mappings_parsed (second element of each pair)
    if pd.notna(row.get('mappings_parsed')):
        try:
            mappings = ast.literal_eval(row['mappings_parsed'])
            for pair in mappings:
                if len(pair) > 1:
                    source_properties_lookup[source_name].add(pair[1])
        except:
            pass

# Convert sets to comma-separated strings
source_properties_lookup = {k: ", ".join(sorted(v)) for k, v in source_properties_lookup.items()}

def get_source_properties(source_name):
    """Get properties for a source from lookup"""
    return source_properties_lookup.get(source_name, "")

def llm_select_sources(target_name, target_properties, candidates, llm_client, top_k=3):
    """Use Llama 405B to select best sources from candidates"""
    
    # Format candidates for prompt with properties from lookup
    candidates_text = "\n".join([
        f"{i+1}. {c.name} - {get_source_properties(c.name)}"
        for i, c in enumerate(candidates)
    ])
    
    prompt = f"""You are selecting analogous sources for a scientific analogy.

Target concept: {target_name}
Target : {target_properties}

Candidate source concepts:
{candidates_text}

Select the {top_k} BEST candidates that would make good analogies for the target concept.
Return ONLY a JSON array of the candidate numbers (1-indexed), e.g., [1, 3, 5]
Do not include any explanation, just the JSON array.
"""
    #print("prompt:\n", prompt)
    
    try:
        response = llm_client.chat(
            model_name="llama-3.1-405b-instruct",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
            max_tokens=100
        )
        # Try to extract JSON array from response
        response_clean = response.strip()
        
        # Try direct JSON parse first
        try:
            selected = json.loads(response_clean)
        except json.JSONDecodeError:
            # Try to find array pattern in response
            match = re.search(r'\[[\d,\s]+\]', response_clean)
            if match:
                selected = json.loads(match.group())
            else:
                # Extract numbers from response
                numbers = re.findall(r'\b([1-9]|10)\b', response_clean)
                selected = [int(n) for n in numbers[:top_k]]
        
        if not selected:
            return candidates[:top_k]
            
        return [candidates[i-1] for i in selected if 1 <= i <= len(candidates)][:top_k]
    except Exception as e:
        print(f"LLM selection failed: {e}, falling back to top {top_k}")
        return candidates[:top_k]


In [ ]:
# Run Baseline 2: Embedding + LLM selection
df = pd.read_csv(DATA_PATH)
baseline2_results = []

# Helper to extract target properties from mappings_parsed
def get_target_properties(row):
    if pd.notna(row.get('mappings_parsed')):
        try:
            mappings = ast.literal_eval(row['mappings_parsed'])
            props = [pair[0] for pair in mappings if len(pair) > 0]
            return ", ".join(props)
        except:
            pass
    return ""

print(f"Running Baseline 2 on {len(df)} examples...")
for idx, row in df.iterrows():
    target_name = row['system_a']
    target_bg = row['system_a_background']
    target_properties = get_target_properties(row)
    gold_source = row['system_b']
    
    # Get top 10 from embeddings
    candidates = finder.find_source(target_name, target_bg, top_k=10)
    # LLM selects best 3
    selected = llm_select_sources(target_name, target_properties, candidates, llm_client, top_k=3)
    
    # Find gold rank in selected
    selected_names = [s.name for s in selected]
    print("target_name:\n", target_name)
    print("selected_names:\n", selected_names)
    gold_rank = -1
    for i, name in enumerate(selected_names):
        if gold_source.lower() in name.lower() or name.lower() in gold_source.lower():
            gold_rank = i + 1
            break
    
    baseline2_results.append({
        'id': row['id'],
        'target': target_name,
        'target_background': target_bg,
        'gold_source': gold_source,
        'top_10_sources': [c.name for c in candidates],
        'top_10_scores': [c.similarity_score for c in candidates],
        'llm_selected_sources': selected_names,
        'llm_selected_scores': [s.similarity_score for s in selected],
        'gold_rank': gold_rank,
        'embedding_mode': finder.embedding_mode,
        'baseline': 'embedding_llm_top10_to_3'
    })
    
    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1}/{len(df)}")

baseline2_df = pd.DataFrame(baseline2_results)
baseline2_df.to_csv('baseline2_embedding_llm.csv', index=False)
print(f"Baseline 2 saved: {len(baseline2_df)} rows")
baseline2_df.head()


Running Baseline 2 on 400 examples...
target_name:
 biological clock
selected_names:
 ['clock', 'day and night cycle', 'Time Machine']
target_name:
 Biosphere
selected_names:
 ['Ecosystem', 'ecosystem', 'Desert']
target_name:
 Respiratory system
selected_names:
 ['Air handling system', 'air circulation ducts', 'lubrication system']
target_name:
 Spread of Pathogens
selected_names:
 ['Spread of Fire', 'Disease', 'Population Movement']
target_name:
 Gene editing
selected_names:
 ['bacterial mutation', 'edit', 'Evolution']
target_name:
 Water Cycle
selected_names:
 ['Wave Propagation', 'Water pipe system', 'Conservation of Water Flow']
target_name:
 Cell division
selected_names:
 ['program', 'Factory', 'Disassembly']
target_name:
 Origin of Life
selected_names:
 ['Evolution', 'Biological Evolution', 'bacterial mutation']
target_name:
 The Genetic Code
selected_names:
 ['Computer Code', 'Deciphering the Code', 'The Neural Network of Computers']
target_name:
 Ecosystem
selected_names:
 ['Ec

,id,target,target_background,gold_source,top_10_sources,top_10_scores,llm_selected_sources,llm_selected_scores,gold_rank,embedding_mode,baseline
0,1,biological clock,The biological clock is a fundamental aspect o...,clock,"[Biological Evolution, clock, bacterial mutati...","[0.3786787182861502, 0.37565001052459696, 0.31...","[clock, day and night cycle, Time Machine]","[0.37565001052459696, 0.29895759869257044, 0.2...",1,name_properties,embedding_llm_top10_to_3
1,2,Biosphere,The biosphere refers to the sum of all ecosyst...,Library,"[Ecosystem, ecosystem, Biological Evolution, C...","[0.5644059383971753, 0.5044757153365504, 0.355...","[Ecosystem, ecosystem, Desert]","[0.5644059383971753, 0.5044757153365504, 0.341...",-1,name_properties,embedding_llm_top10_to_3
2,3,Respiratory system,The respiratory system is a critical biologica...,engine,"[respiration, Air handling system, Human Body,...","[0.4439990903135665, 0.40105959603856384, 0.39...","[Air handling system, air circulation ducts, l...","[0.40105959603856384, 0.3127274941224897, 0.29...",-1,name_properties,embedding_llm_top10_to_3
3,4,Spread of Pathogens,"The spread of pathogens is a serious concern, ...",Spread of Fire,"[Spread of Fire, Disease, Population Movement,...","[0.44654289141485515, 0.43497819694422885, 0.3...","[Spread of Fire, Disease, Population Movement]","[0.44654289141485515, 0.43497819694422885, 0.3...",1,name_properties,embedding_llm_top10_to_3
4,5,Gene editing,Gene editing is a revolutionary technique in m...,kirigami,"[bacterial mutation, edit, Evolution, program,...","[0.3618950292759755, 0.3256726784890958, 0.315...","[bacterial mutation, edit, Evolution]","[0.3618950292759755, 0.3256726784890958, 0.315...",-1,name_properties,embedding_llm_top10_to_3


## Baseline 2b: LLM Selection - Target Name Only


In [24]:
def llm_select_sources_name_only(target_name, candidates, llm_client, top_k=3):
    """Use Llama 405B - Target name only"""
    candidates_text = "\n".join([
        f"{i+1}. {c.name}"
        for i, c in enumerate(candidates)
    ])
    
    prompt = f"""You are selecting analogous sources for a scientific analogy.

Target concept: {target_name}

Candidate source concepts:
{candidates_text}

Select the {top_k} BEST candidates that would make good analogies for the target concept.
Return ONLY a JSON array of the candidate numbers (1-indexed), e.g., [1, 3, 5]
Do not include any explanation, just the JSON array.
"""
    
    try:
        response = llm_client.chat(
            model_name="llama-3.1-405b-instruct",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
            max_tokens=100
        )
        response_clean = response.strip()
        try:
            selected = json.loads(response_clean)
        except json.JSONDecodeError:
            match = re.search(r'\[[\d,\s]+\]', response_clean)
            if match:
                selected = json.loads(match.group())
            else:
                numbers = re.findall(r'\b([1-9]|10)\b', response_clean)
                selected = [int(n) for n in numbers[:top_k]]
        if not selected:
            return candidates[:top_k]
        return [candidates[i-1] for i in selected if 1 <= i <= len(candidates)][:top_k]
    except Exception as e:
        print(f"LLM selection failed: {e}, falling back to top {top_k}")
        return candidates[:top_k]

# Run Baseline 2b
baseline2b_results = []
print(f"Running Baseline 2b (name only) on {len(df)} examples...")
for idx, row in df.iterrows():
    target_name = row['system_a']
    target_bg = row['system_a_background']
    gold_source = row['system_b']
    
    candidates = finder.find_source(target_name, target_bg, top_k=10)
    selected = llm_select_sources_name_only(target_name, candidates, llm_client, top_k=3)
    
    selected_names = [s.name for s in selected]
    gold_rank = -1
    for i, name in enumerate(selected_names):
        if gold_source.lower() in name.lower() or name.lower() in gold_source.lower():
            gold_rank = i + 1
            break
    
    baseline2b_results.append({
        'id': row['id'],
        'target': target_name,
        'gold_source': gold_source,
        'llm_selected_sources': selected_names,
        'gold_rank': gold_rank,
        'condition': 'name_only'
    })
    
    if (idx + 1) % 50 == 0:
        print(f"Processed {idx + 1}/{len(df)}")

baseline2b_df = pd.DataFrame(baseline2b_results)
baseline2b_df.to_csv('baseline2b_name_only.csv', index=False)
print(f"Baseline 2b saved: {len(baseline2b_df)} rows")


Running Baseline 2b (name only) on 400 examples...
Processed 50/400
Processed 100/400
Processed 150/400
Processed 200/400
Processed 250/400
Processed 300/400
Processed 350/400
Processed 400/400
Baseline 2b saved: 400 rows


## Baseline 2c: LLM Selection - Target Name + Properties + Description


In [25]:
# Build source description lookup
source_desc_lookup = {}
for _, row in df_scar.iterrows():
    source_name = row['system_b']
    if source_name not in source_desc_lookup:
        source_desc_lookup[source_name] = row['system_b_background'] if pd.notna(row['system_b_background']) else ""

def llm_select_sources_full(target_name, target_properties, target_desc, candidates, llm_client, top_k=3):
    """Use Llama 405B - Target name + properties + description"""
    candidates_text = "\n".join([
        f"{i+1}. {c.name} - Properties: {get_source_properties(c.name)} - Description: {source_desc_lookup.get(c.name, '')[:200]}..."
        for i, c in enumerate(candidates)
    ])
    
    prompt = f"""You are selecting analogous sources for a scientific analogy.

Target concept: {target_name}
Target properties: {target_properties}
Target description: {target_desc}

Candidate source concepts:
{candidates_text}

Select the {top_k} BEST candidates that would make good analogies for the target concept.
Return ONLY a JSON array of the candidate numbers (1-indexed), e.g., [1, 3, 5]
Do not include any explanation, just the JSON array.
"""
    
    try:
        response = llm_client.chat(
            model_name="llama-3.1-405b-instruct",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
            max_tokens=100
        )
        response_clean = response.strip()
        try:
            selected = json.loads(response_clean)
        except json.JSONDecodeError:
            match = re.search(r'\[[\d,\s]+\]', response_clean)
            if match:
                selected = json.loads(match.group())
            else:
                numbers = re.findall(r'\b([1-9]|10)\b', response_clean)
                selected = [int(n) for n in numbers[:top_k]]
        if not selected:
            return candidates[:top_k]
        return [candidates[i-1] for i in selected if 1 <= i <= len(candidates)][:top_k]
    except Exception as e:
        print(f"LLM selection failed: {e}, falling back to top {top_k}")
        return candidates[:top_k]

# Run Baseline 2c
baseline2c_results = []
print(f"Running Baseline 2c (name + properties + description) on {len(df)} examples...")
for idx, row in df.iterrows():
    target_name = row['system_a']
    target_bg = row['system_a_background']
    target_properties = get_target_properties(row)
    gold_source = row['system_b']
    
    candidates = finder.find_source(target_name, target_bg, top_k=10)
    selected = llm_select_sources_full(target_name, target_properties, target_bg, candidates, llm_client, top_k=3)
    
    selected_names = [s.name for s in selected]
    gold_rank = -1
    for i, name in enumerate(selected_names):
        if gold_source.lower() in name.lower() or name.lower() in gold_source.lower():
            gold_rank = i + 1
            break
    
    baseline2c_results.append({
        'id': row['id'],
        'target': target_name,
        'gold_source': gold_source,
        'llm_selected_sources': selected_names,
        'gold_rank': gold_rank,
        'condition': 'name_properties_description'
    })
    
    if (idx + 1) % 50 == 0:
        print(f"Processed {idx + 1}/{len(df)}")

baseline2c_df = pd.DataFrame(baseline2c_results)
baseline2c_df.to_csv('baseline2c_full.csv', index=False)
print(f"Baseline 2c saved: {len(baseline2c_df)} rows")


Running Baseline 2c (name + properties + description) on 400 examples...
Processed 50/400
Processed 100/400
Processed 150/400
Processed 200/400
Processed 250/400
Processed 300/400
Processed 350/400
Processed 400/400
Baseline 2c saved: 400 rows


## Baseline 2d: LLM Selection - Target Name + Description


In [26]:
def llm_select_sources_name_desc(target_name, target_desc, candidates, llm_client, top_k=3):
    """Use Llama 405B - Target name + description"""
    candidates_text = "\n".join([
        f"{i+1}. {c.name} - Description: {source_desc_lookup.get(c.name, '')[:200]}..."
        for i, c in enumerate(candidates)
    ])
    
    prompt = f"""You are selecting analogous sources for a scientific analogy.

Target concept: {target_name}
Target description: {target_desc}

Candidate source concepts:
{candidates_text}

Select the {top_k} BEST candidates that would make good analogies for the target concept.
Return ONLY a JSON array of the candidate numbers (1-indexed), e.g., [1, 3, 5]
Do not include any explanation, just the JSON array.
"""
    
    try:
        response = llm_client.chat(
            model_name="llama-3.1-405b-instruct",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
            max_tokens=100
        )
        response_clean = response.strip()
        try:
            selected = json.loads(response_clean)
        except json.JSONDecodeError:
            match = re.search(r'\[[\d,\s]+\]', response_clean)
            if match:
                selected = json.loads(match.group())
            else:
                numbers = re.findall(r'\b([1-9]|10)\b', response_clean)
                selected = [int(n) for n in numbers[:top_k]]
        if not selected:
            return candidates[:top_k]
        return [candidates[i-1] for i in selected if 1 <= i <= len(candidates)][:top_k]
    except Exception as e:
        print(f"LLM selection failed: {e}, falling back to top {top_k}")
        return candidates[:top_k]

# Run Baseline 2d
baseline2d_results = []
print(f"Running Baseline 2d (name + description) on {len(df)} examples...")
for idx, row in df.iterrows():
    target_name = row['system_a']
    target_bg = row['system_a_background']
    gold_source = row['system_b']
    
    candidates = finder.find_source(target_name, target_bg, top_k=10)
    selected = llm_select_sources_name_desc(target_name, target_bg, candidates, llm_client, top_k=3)
    
    selected_names = [s.name for s in selected]
    gold_rank = -1
    for i, name in enumerate(selected_names):
        if gold_source.lower() in name.lower() or name.lower() in gold_source.lower():
            gold_rank = i + 1
            break
    
    baseline2d_results.append({
        'id': row['id'],
        'target': target_name,
        'gold_source': gold_source,
        'llm_selected_sources': selected_names,
        'gold_rank': gold_rank,
        'condition': 'name_description'
    })
    
    if (idx + 1) % 50 == 0:
        print(f"Processed {idx + 1}/{len(df)}")

baseline2d_df = pd.DataFrame(baseline2d_results)
baseline2d_df.to_csv('baseline2d_name_desc.csv', index=False)
print(f"Baseline 2d saved: {len(baseline2d_df)} rows")


Running Baseline 2d (name + description) on 400 examples...
Processed 50/400
Processed 100/400
Processed 150/400
Processed 200/400
Processed 250/400
Processed 300/400
Processed 350/400
Processed 400/400
Baseline 2d saved: 400 rows


## Summary Statistics


In [27]:
# Compare all baselines
print("=" * 60)
print("BASELINE COMPARISON")
print("=" * 60)

results_summary = []

# Baseline 1 metrics
b1_found = (baseline1_results['gold_rank'] > 0).sum()
b1_total = len(baseline1_results)
b1_mean = baseline1_results[baseline1_results['gold_rank'] > 0]['gold_rank'].mean()
print(f"\nBaseline 1 (Embedding Top 3):")
print(f"  Gold found: {b1_found}/{b1_total} ({100*b1_found/b1_total:.1f}%)")
print(f"  Mean gold rank (when found): {b1_mean:.2f}")
results_summary.append({'baseline': 'Embedding Top 3', 'found': b1_found, 'total': b1_total, 'hit_rate': 100*b1_found/b1_total, 'mean_rank': b1_mean})

# Baseline 2 metrics (properties only)
b2_found = (baseline2_df['gold_rank'] > 0).sum()
b2_total = len(baseline2_df)
b2_mean = baseline2_df[baseline2_df['gold_rank'] > 0]['gold_rank'].mean()
print(f"\nBaseline 2 (LLM + Properties Only):")
print(f"  Gold found: {b2_found}/{b2_total} ({100*b2_found/b2_total:.1f}%)")
print(f"  Mean gold rank (when found): {b2_mean:.2f}")
results_summary.append({'baseline': 'LLM + Properties Only', 'found': b2_found, 'total': b2_total, 'hit_rate': 100*b2_found/b2_total, 'mean_rank': b2_mean})

# Baseline 2b metrics (name only)
b2b_found = (baseline2b_df['gold_rank'] > 0).sum()
b2b_total = len(baseline2b_df)
b2b_mean = baseline2b_df[baseline2b_df['gold_rank'] > 0]['gold_rank'].mean()
print(f"\nBaseline 2b (LLM + Name Only):")
print(f"  Gold found: {b2b_found}/{b2b_total} ({100*b2b_found/b2b_total:.1f}%)")
print(f"  Mean gold rank (when found): {b2b_mean:.2f}")
results_summary.append({'baseline': 'LLM + Name Only', 'found': b2b_found, 'total': b2b_total, 'hit_rate': 100*b2b_found/b2b_total, 'mean_rank': b2b_mean})

# Baseline 2c metrics (full)
b2c_found = (baseline2c_df['gold_rank'] > 0).sum()
b2c_total = len(baseline2c_df)
b2c_mean = baseline2c_df[baseline2c_df['gold_rank'] > 0]['gold_rank'].mean()
print(f"\nBaseline 2c (LLM + Name + Properties + Description):")
print(f"  Gold found: {b2c_found}/{b2c_total} ({100*b2c_found/b2c_total:.1f}%)")
print(f"  Mean gold rank (when found): {b2c_mean:.2f}")
results_summary.append({'baseline': 'LLM: Name + Props + Desc', 'found': b2c_found, 'total': b2c_total, 'hit_rate': 100*b2c_found/b2c_total, 'mean_rank': b2c_mean})

# Baseline 2d metrics (name + description)
b2d_found = (baseline2d_df['gold_rank'] > 0).sum()
b2d_total = len(baseline2d_df)
b2d_mean = baseline2d_df[baseline2d_df['gold_rank'] > 0]['gold_rank'].mean()
print(f"\nBaseline 2d (LLM + Name + Description):")
print(f"  Gold found: {b2d_found}/{b2d_total} ({100*b2d_found/b2d_total:.1f}%)")
print(f"  Mean gold rank (when found): {b2d_mean:.2f}")
results_summary.append({'baseline': 'LLM: Name + Desc', 'found': b2d_found, 'total': b2d_total, 'hit_rate': 100*b2d_found/b2d_total, 'mean_rank': b2d_mean})

print("\n" + "=" * 60)
summary_df = pd.DataFrame(results_summary)
summary_df.to_csv('baselines_summary.csv', index=False)
print("\nSummary saved to baselines_summary.csv")
summary_df


BASELINE COMPARISON

Baseline 1 (Embedding Top 3):
  Gold found: 160/400 (40.0%)
  Mean gold rank (when found): 1.75

Baseline 2 (LLM + Properties Only):
  Gold found: 145/400 (36.2%)
  Mean gold rank (when found): 1.66

Baseline 2b (LLM + Name Only):
  Gold found: 129/400 (32.2%)
  Mean gold rank (when found): 1.65

Baseline 2c (LLM + Name + Properties + Description):
  Gold found: 134/400 (33.5%)
  Mean gold rank (when found): 1.55

Baseline 2d (LLM + Name + Description):
  Gold found: 130/400 (32.5%)
  Mean gold rank (when found): 1.62


Summary saved to baselines_summary.csv


,baseline,found,total,hit_rate,mean_rank
0,Embedding Top 3,160,400,40.00,1.750000
1,LLM + Properties Only,145,400,36.25,1.655172
2,LLM + Name Only,129,400,32.25,1.651163
3,LLM: Name + Props + Desc,134,400,33.50,1.552239
4,LLM: Name + Desc,130,400,32.50,1.615385
